# 2.2 不确定性评估
本 Notebook 基于 **PyTorch Geometric (PyG)**，演示不确定性方法在分子性质预测中的应用：
- **MC-Dropout**
- **EDL**

### （1） 训练GIN模型，用于后续MC-Dropout分析

In [22]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool
from torch.utils.data import random_split
import torch_geometric.transforms as T
from torch_geometric.data import Data
from torch_geometric.datasets import MoleculeNet
from rdkit import Chem
import copy
from rdkit.Chem import Draw
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import SimilarityMaps
from IPython.display import SVG, display
from torch_geometric.utils.smiles import from_smiles
from rdkit.Chem.Draw import rdMolDraw2D
import os
import time
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch_geometric.loader import DataLoader


# =========================================================
# 0. 运行前提
# =========================================================
# 你需要在前面已经准备好：
# - ds_cls : 分类数据集（如 BBBP）
# - ds_reg : 回归数据集（如 ESOL）
# - device : 运行设备，例如：
#   device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================================================
# 1. 固定随机种子
# =========================================================
def set_global_seed(seed=42, deterministic=True):
    """固定随机种子，尽量保证结果可复现"""
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass


# =========================================================
# 2. 数据集划分
# =========================================================
def split_dataset(dataset, train_ratio=0.8, val_ratio=0.1, seed=42):
    """
    将数据集划分为 train / val / test
    默认比例：8 : 1 : 1
    """
    n = len(dataset)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    n_test = n - n_train - n_val

    generator = torch.Generator().manual_seed(seed)
    train_set, val_set, test_set = random_split(
        dataset,
        [n_train, n_val, n_test],
        generator=generator
    )
    return train_set, val_set, test_set


# =========================================================
# 3. 超参数配置
# =========================================================
CLS_HP = {
    "task": "cls",
    "seed": 42,
    "batch_size_train": 64,
    "batch_size_eval": 256,
    "hidden_dim": 128,
    "num_layers": 4,
    "dropout": 0.2,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "epochs": 100,
    "model_dir": "saved_models",
    "task_name": "gin_cls"
}

REG_HP = {
    "task": "reg",
    "seed": 42,
    "batch_size_train": 64,
    "batch_size_eval": 256,
    "hidden_dim": 128,
    "num_layers": 4,
    "dropout": 0.2,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "epochs": 100,
    "model_dir": "saved_models",
    "task_name": "gin_reg"
}


# =========================================================
# 4. 模型定义
# =========================================================
class MLP(nn.Module):
    """两层感知机，用作分类头或回归头"""
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class GINBackbone(nn.Module):
    """GIN 主干：输入图，输出图级表示"""
    def __init__(self, in_dim, hidden_dim=128, num_layers=4, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()

        for i in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(in_dim if i == 0 else hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        g = global_mean_pool(x, batch)
        return g


class GINClassifier(nn.Module):
    """GIN 二分类模型，输出 2 维 logits"""
    def __init__(self, in_dim, hidden_dim=128, num_layers=4, dropout=0.2):
        super().__init__()
        self.gnn = GINBackbone(in_dim, hidden_dim, num_layers, dropout)
        self.head = MLP(hidden_dim, hidden_dim, 2, dropout)

    def forward(self, data):
        g = self.gnn(data.x, data.edge_index, data.batch)
        return self.head(g)


class GINRegressor(nn.Module):
    """GIN 回归模型，输出 1 个连续值"""
    def __init__(self, in_dim, hidden_dim=128, num_layers=4, dropout=0.2):
        super().__init__()
        self.gnn = GINBackbone(in_dim, hidden_dim, num_layers, dropout)
        self.head = MLP(hidden_dim, hidden_dim, 1, dropout)

    def forward(self, data):
        g = self.gnn(data.x, data.edge_index, data.batch)
        return self.head(g).view(-1)


def build_model(task, dataset, hp, device):
    """根据任务类型构建模型"""
    if task == "cls":
        return GINClassifier(
            in_dim=dataset.num_features,
            hidden_dim=hp["hidden_dim"],
            num_layers=hp["num_layers"],
            dropout=hp["dropout"]
        ).to(device)

    if task == "reg":
        return GINRegressor(
            in_dim=dataset.num_features,
            hidden_dim=hp["hidden_dim"],
            num_layers=hp["num_layers"],
            dropout=hp["dropout"]
        ).to(device)

    raise ValueError(f"未知任务类型：{task}")


# =========================================================
# 5. 训练与评估函数
# =========================================================
def train_one_epoch_classification(model, loader, optimizer, device):
    """分类任务训练一个 epoch，返回平均交叉熵损失"""
    model.train()
    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)

        logits = model(batch)
        y = batch.y.view(-1).long()
        loss = F.cross_entropy(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.detach().item() * y.size(0)

    return total_loss / max(len(loader.dataset), 1)


def train_one_epoch_regression(model, loader, optimizer, device):
    """回归任务训练一个 epoch，返回平均 MSE 损失"""
    model.train()
    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)

        pred = model(batch)
        y = batch.y.view(-1).float()
        loss = F.mse_loss(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.detach().item() * y.size(0)

    return total_loss / max(len(loader.dataset), 1)


@torch.no_grad()
def eval_classification(model, loader, device):
    """分类评估：返回 accuracy"""
    model.eval()
    correct, total = 0, 0

    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        pred = logits.argmax(dim=-1)
        y = batch.y.view(-1).long()

        correct += int((pred == y).sum())
        total += y.numel()

    return correct / max(total, 1)


@torch.no_grad()
def eval_regression(model, loader, device):
    """回归评估：返回 (mae, rmse)"""
    model.eval()
    ys, ps = [], []

    for batch in loader:
        batch = batch.to(device)
        pred = model(batch)
        y = batch.y.view(-1).float()

        ys.append(y.detach().cpu())
        ps.append(pred.detach().cpu())

    y = torch.cat(ys)
    p = torch.cat(ps)

    mae = torch.mean(torch.abs(p - y)).item()
    rmse = torch.sqrt(torch.mean((p - y) ** 2)).item()
    return mae, rmse


# =========================================================
# 6. 保存与加载
# =========================================================
def make_model_name(task_name, hp):
    """根据超参数生成模型文件名"""
    return (
        f"{task_name}"
        f"_seed{hp['seed']}"
        f"_bs{hp['batch_size_train']}"
        f"_hd{hp['hidden_dim']}"
        f"_L{hp['num_layers']}"
        f"_do{hp['dropout']}"
        f"_lr{hp['lr']}"
        f"_wd{hp['weight_decay']}"
        f"_ep{hp['epochs']}"
        f".pt"
    )


def load_trained_model(task, dataset, hp, device):
    """加载已保存模型"""
    if task == "cls":
        task_name = "gin_cls"
    elif task == "reg":
        task_name = "gin_reg"
    else:
        raise ValueError(f"未知任务类型：{task}")

    save_name = make_model_name(task_name, hp)
    save_path = os.path.join(hp["model_dir"], save_name)

    if not os.path.exists(save_path):
        raise FileNotFoundError(f"找不到模型文件：{save_path}")

    model = build_model(task, dataset, hp, device)
    ckpt = torch.load(save_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    print(f"成功加载模型：{save_path}")
    return model, ckpt


# =========================================================
# 7. 分类训练主函数
# =========================================================
def train_classifier(dataset, hp, device):
    """
    分类训练主函数
    模型选择标准：验证集 accuracy 最大
    """
    set_global_seed(hp["seed"])

    train_set, val_set, test_set = split_dataset(dataset, seed=hp["seed"])

    train_loader = DataLoader(
        train_set,
        batch_size=hp["batch_size_train"],
        shuffle=True
    )
    val_loader = DataLoader(
        val_set,
        batch_size=hp["batch_size_eval"],
        shuffle=False
    )
    test_loader = DataLoader(
        test_set,
        batch_size=hp["batch_size_eval"],
        shuffle=False
    )

    model = build_model("cls", dataset, hp, device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=hp["lr"],
        weight_decay=hp["weight_decay"]
    )

    best_val_acc = 0.0
    best_state = None

    print("\n开始训练分类模型...")

    for epoch in range(1, hp["epochs"] + 1):
        train_loss = train_one_epoch_classification(model, train_loader, optimizer, device)
        val_acc = eval_classification(model, val_loader, device)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"第 {epoch:03d} 轮 | "
                f"训练损失 = {train_loss:.4f} | "
                f"验证集准确率 = {val_acc:.4f}"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    test_acc = eval_classification(model, test_loader, device)

    Path(hp["model_dir"]).mkdir(parents=True, exist_ok=True)
    save_name = make_model_name(hp["task_name"], hp)
    save_path = os.path.join(hp["model_dir"], save_name)

    torch.save({
        "task": "cls",
        "model_state_dict": model.state_dict(),
        "hyperparameters": dict(hp),
        "best_val_acc": best_val_acc,
        "test_acc": test_acc
    }, save_path)

    print(f"\n分类模型训练完成")
    print(f"最佳验证集准确率 = {best_val_acc:.4f}")
    print(f"测试集准确率     = {test_acc:.4f}")
    print(f"模型已保存到：{save_path}")

    return model, save_path


# =========================================================
# 8. 回归训练主函数
# =========================================================
def train_regressor(dataset, hp, device):
    """
    回归训练主函数
    模型选择标准：验证集 MAE 最小
    """
    set_global_seed(hp["seed"])

    train_set, val_set, test_set = split_dataset(dataset, seed=hp["seed"])

    train_loader = DataLoader(
        train_set,
        batch_size=hp["batch_size_train"],
        shuffle=True
    )
    val_loader = DataLoader(
        val_set,
        batch_size=hp["batch_size_eval"],
        shuffle=False
    )
    test_loader = DataLoader(
        test_set,
        batch_size=hp["batch_size_eval"],
        shuffle=False
    )

    model = build_model("reg", dataset, hp, device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=hp["lr"],
        weight_decay=hp["weight_decay"]
    )

    best_val_mae = float("inf")
    best_state = None

    print("\n开始训练回归模型...")

    for epoch in range(1, hp["epochs"] + 1):
        train_loss = train_one_epoch_regression(model, train_loader, optimizer, device)
        val_mae, val_rmse = eval_regression(model, val_loader, device)

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"第 {epoch:03d} 轮 | "
                f"训练损失 = {train_loss:.4f} | "
                f"验证集 MAE = {val_mae:.4f} | "
                f"验证集 RMSE = {val_rmse:.4f}"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    test_mae, test_rmse = eval_regression(model, test_loader, device)

    Path(hp["model_dir"]).mkdir(parents=True, exist_ok=True)
    save_name = make_model_name(hp["task_name"], hp)
    save_path = os.path.join(hp["model_dir"], save_name)

    torch.save({
        "task": "reg",
        "model_state_dict": model.state_dict(),
        "hyperparameters": dict(hp),
        "best_val_mae": best_val_mae,
        "test_mae": test_mae,
        "test_rmse": test_rmse
    }, save_path)

    print(f"\n回归模型训练完成")
    print(f"最佳验证集 MAE = {best_val_mae:.4f}")
    print(f"测试集 MAE     = {test_mae:.4f}")
    print(f"测试集 RMSE    = {test_rmse:.4f}")
    print(f"模型已保存到：{save_path}")

    return model, save_path


# =========================================================
# 9. 调用示例
# =========================================================
def load_bbbp(root="./data"):
    ds = MoleculeNet(root=root, name="BBBP")
    return ds, "BBBP"


def load_esol(root="./data"):
    ds = MoleculeNet(root=root, name="ESOL")
    return ds, "ESOL"

# 载入数据
ds_cls, cls_name = load_bbbp("./data")
ds_reg, reg_name = load_esol("./data")

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed_everything(42)
device = get_device()

# 分类训练（例如 BBBP）
model_cls, cls_save_path = train_classifier(ds_cls, CLS_HP, device)

# 回归训练（例如 ESOL）
model_reg, reg_save_path = train_regressor(ds_reg, REG_HP, device)


开始训练分类模型...
第 001 轮 | 训练损失 = 0.4913 | 验证集准确率 = 0.7192
第 010 轮 | 训练损失 = 0.3562 | 验证集准确率 = 0.8177
第 020 轮 | 训练损失 = 0.2993 | 验证集准确率 = 0.7241
第 030 轮 | 训练损失 = 0.2544 | 验证集准确率 = 0.8177
第 040 轮 | 训练损失 = 0.2342 | 验证集准确率 = 0.8621
第 050 轮 | 训练损失 = 0.2366 | 验证集准确率 = 0.8325
第 060 轮 | 训练损失 = 0.2179 | 验证集准确率 = 0.8522
第 070 轮 | 训练损失 = 0.1806 | 验证集准确率 = 0.8374
第 080 轮 | 训练损失 = 0.1706 | 验证集准确率 = 0.8571
第 090 轮 | 训练损失 = 0.1490 | 验证集准确率 = 0.8571
第 100 轮 | 训练损失 = 0.1453 | 验证集准确率 = 0.8768

分类模型训练完成
最佳验证集准确率 = 0.8867
测试集准确率     = 0.8293
模型已保存到：saved_models\gin_cls_seed42_bs64_hd128_L4_do0.2_lr0.001_wd1e-05_ep100.pt

开始训练回归模型...
第 001 轮 | 训练损失 = 8.6798 | 验证集 MAE = 1.6886 | 验证集 RMSE = 2.2202
第 010 轮 | 训练损失 = 1.3048 | 验证集 MAE = 1.1773 | 验证集 RMSE = 1.6405
第 020 轮 | 训练损失 = 1.0854 | 验证集 MAE = 1.0389 | 验证集 RMSE = 1.3597
第 030 轮 | 训练损失 = 1.0298 | 验证集 MAE = 0.9065 | 验证集 RMSE = 1.1774
第 040 轮 | 训练损失 = 0.9985 | 验证集 MAE = 0.9824 | 验证集 RMSE = 1.3094
第 050 轮 | 训练损失 = 0.8493 | 验证集 MAE = 0.9336 | 验证集 RMSE = 1.1949
第 060 

### （2）MC-Dropout

In [23]:
# =========================
# 0. 不确定度评估参数区
#   主要修改这里，也决定载入的模型参数
# =========================
UQ_CFG = {
    "task": "cls",                 # 任务类型："cls" 表示分类，"reg" 表示回归
    "seed": 42,                    # 随机种子
    "batch_size_eval": 64,         # 评估时的 batch 大小
    "hidden_dim": 128,             # 隐藏层维度
    "num_layers": 4,               # GINConv 层数
    "dropout": 0.2,                # Dropout 概率
    "lr": 1e-3,                    # 学习率
    "weight_decay": 1e-5,          # 权重衰减
    "epochs": 100,                 # 训练轮数（用于匹配模型文件名）
    "model_dir": "saved_models",   # 模型保存目录
    "mc_T": 100,                    # MC-Dropout 采样次数
    "eval_split": "test"           # 评估数据划分："train" / "val" / "test" / "all"
}


# =========================
# 1. 数据集划分函数
# =========================
def split_dataset(dataset, train_ratio=0.8, val_ratio=0.1, seed=42):
    """
    将数据集划分为训练集 / 验证集 / 测试集
    """
    n = len(dataset)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    n_test = n - n_train - n_val

    generator = torch.Generator().manual_seed(seed)
    train_set, val_set, test_set = random_split(
        dataset,
        [n_train, n_val, n_test],
        generator=generator
    )
    return train_set, val_set, test_set


# =========================
# 2. 根据超参数生成模型文件名
# =========================
def make_model_name(task_name, hp):
    """
    根据超参数生成模型文件名
    注意：这里默认训练时 batch_size_train 固定写成 64，
    因此文件名中仍写死为 _bs64。
    """
    return (
        f"{task_name}"
        f"_seed{hp['seed']}"
        f"_bs64"
        f"_hd{hp['hidden_dim']}"
        f"_L{hp['num_layers']}"
        f"_do{hp['dropout']}"
        f"_lr{hp['lr']}"
        f"_wd{hp['weight_decay']}"
        f"_ep{hp['epochs']}"
        f".pt"
    )


# =========================
# 3. 根据任务构建模型
# =========================
def build_model(task, dataset, cfg, device):
    """
    根据任务类型构建模型
    """
    if task == "cls":
        model = GINClassifier(
            in_dim=dataset.num_features,
            hidden_dim=cfg["hidden_dim"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"]
        ).to(device)

    elif task == "reg":
        model = GINRegressor(
            in_dim=dataset.num_features,
            hidden_dim=cfg["hidden_dim"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"]
        ).to(device)

    else:
        raise ValueError(f"未知任务类型：{task}")

    return model


# =========================
# 4. 加载已经训练好的模型
# =========================
def load_trained_model(task, dataset, cfg, device):
    """
    根据超参数自动定位并加载已训练模型
    """
    task_name = "gin_cls" if task == "cls" else "gin_reg"
    save_name = make_model_name(task_name, cfg)
    save_path = os.path.join(cfg["model_dir"], save_name)

    if not os.path.exists(save_path):
        raise FileNotFoundError(f"找不到模型文件：{save_path}")

    model = build_model(task, dataset, cfg, device)

    ckpt = torch.load(save_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])

    print(f"成功加载模型：{save_path}")
    if "hyperparameters" in ckpt:
        print("Checkpoint 中记录的超参数：")
        for k, v in ckpt["hyperparameters"].items():
            print(f"  {k}: {v}")

    return model, ckpt


# =========================
# 5. 根据数据划分方式选择评估集
# =========================
def get_eval_loader(dataset, cfg):
    """
    根据 eval_split 选择评估数据
    """
    train_set, val_set, test_set = split_dataset(dataset, seed=cfg["seed"])

    split = cfg["eval_split"]
    if split == "train":
        eval_set = train_set
    elif split == "val":
        eval_set = val_set
    elif split == "test":
        eval_set = test_set
    elif split == "all":
        eval_set = dataset
    else:
        raise ValueError(f"未知的数据划分方式：{split}")

    loader = DataLoader(
        eval_set,
        batch_size=cfg["batch_size_eval"],
        shuffle=False
    )
    return loader


# =========================
# 6. MC-Dropout：分类任务预测
# =========================
def enable_dropout_only(model):
    for m in model.modules():
        if isinstance(m, torch.nn.Dropout):
            m.train()

@torch.no_grad()
def mc_dropout_predict_classification(model, data, T=30):
    """
    分类任务下的 MC-Dropout 预测

    输入：
      - model: 分类模型，输出形状为 [B, 2]
      - data : 一个 batch 的图数据
      - T    : 采样次数

    输出：
      - mean_prob: 平均预测概率，[B, 2]
      - var_prob : 预测概率方差，[B, 2]
    """
    model.eval()
    enable_dropout_only(model)
    probs = []

    for _ in range(T):
        logits = model(data)   # [B, 2]
        probs.append(F.softmax(logits, dim=-1).unsqueeze(0))  # [1, B, 2]

    probs = torch.cat(probs, dim=0)  # [T, B, 2]
    mean_prob = probs.mean(dim=0)
    var_prob = probs.var(dim=0, unbiased=False)
    return mean_prob, var_prob


# =========================
# 7. MC-Dropout：回归任务预测
# =========================
@torch.no_grad()
def mc_dropout_predict_regression(model, data, T=30):
    """
    回归任务下的 MC-Dropout 预测

    输入：
      - model: 回归模型，输出形状为 [B]
      - data : 一个 batch 的图数据
      - T    : 采样次数

    输出：
      - mean_pred: 平均预测值，[B]
      - var_pred : 预测方差，[B]
    """
    model.eval()
    enable_dropout_only(model)
    preds = []

    for _ in range(T):
        preds.append(model(data).view(-1).unsqueeze(0))   # [1, B]

    preds = torch.cat(preds, dim=0)   # [T, B]
    mean_pred = preds.mean(dim=0)
    var_pred = preds.var(dim=0, unbiased=False)
    return mean_pred, var_pred


# =========================
# 8. 分类任务：加载模型并评估不确定度
# =========================
def evaluate_uncertainty_classification(dataset, cfg, device):
    """
    对分类任务进行不确定度评估
    """
    model, ckpt = load_trained_model("cls", dataset, cfg, device)
    loader = get_eval_loader(dataset, cfg)

    cls_means = []
    cls_vars = []
    cls_trues = []

    for batch in loader:
        batch = batch.to(device)

        mean_prob, var_prob = mc_dropout_predict_classification(
            model, batch, T=cfg["mc_T"]
        )

        unc = var_prob.sum(dim=-1).detach().cpu().numpy()   # 每个样本的不确定度
        y_true = batch.y.view(-1).detach().cpu().numpy()

        cls_means.append(mean_prob.detach().cpu().numpy())
        cls_vars.append(unc)
        cls_trues.append(y_true)

    cls_means = np.concatenate(cls_means, axis=0)
    cls_vars = np.concatenate(cls_vars, axis=0)
    cls_trues = np.concatenate(cls_trues, axis=0)

    cls_sorted_idx = np.argsort(cls_vars)
    n_cls = len(cls_sorted_idx)
    cls_group_size = n_cls // 5

    print("\n=== 分类任务：按不确定度分为 5 组 ===")
    for i in range(5):
        start = i * cls_group_size
        end = n_cls if i == 4 else (i + 1) * cls_group_size
        idxs = cls_sorted_idx[start:end]

        preds = cls_means[idxs].argmax(axis=-1)
        trues = cls_trues[idxs]
        acc = (preds == trues).mean()

        print(
            f"第 {i+1:02d} 组 | "
            f"不确定度范围 [{cls_vars[idxs].min():.6f}, {cls_vars[idxs].max():.6f}] | "
            f"样本数 = {len(idxs)} | "
            f"准确率 = {acc:.4f}"
        )

    return {
        "mean_prob": cls_means,
        "uncertainty": cls_vars,
        "y_true": cls_trues
    }


# =========================
# 9. 回归任务：加载模型并评估不确定度
# =========================
def evaluate_uncertainty_regression(dataset, cfg, device):
    """
    对回归任务进行不确定度评估
    """
    model, ckpt = load_trained_model("reg", dataset, cfg, device)
    loader = get_eval_loader(dataset, cfg)

    reg_means = []
    reg_vars = []
    reg_trues = []

    for batch in loader:
        batch = batch.to(device)

        mean_pred, var_pred = mc_dropout_predict_regression(
            model, batch, T=cfg["mc_T"]
        )

        unc = var_pred.detach().cpu().numpy()
        y_true = batch.y.view(-1).detach().cpu().numpy()

        reg_means.append(mean_pred.detach().cpu().numpy())
        reg_vars.append(unc)
        reg_trues.append(y_true)

    reg_means = np.concatenate(reg_means, axis=0)
    reg_vars = np.concatenate(reg_vars, axis=0)
    reg_trues = np.concatenate(reg_trues, axis=0)

    reg_sorted_idx = np.argsort(reg_vars)
    n_reg = len(reg_sorted_idx)
    reg_group_size = n_reg // 5

    print("\n=== 回归任务：按不确定度分为 5 组 ===")
    for i in range(5):
        start = i * reg_group_size
        end = n_reg if i == 4 else (i + 1) * reg_group_size
        idxs = reg_sorted_idx[start:end]

        preds = reg_means[idxs]
        trues = reg_trues[idxs]

        rmse = np.sqrt(((preds - trues) ** 2).mean())
        mae = np.abs(preds - trues).mean()

        print(
            f"第 {i+1:02d} 组 | "
            f"不确定度范围 [{reg_vars[idxs].min():.6f}, {reg_vars[idxs].max():.6f}] | "
            f"样本数 = {len(idxs)} | "
            f"RMSE = {rmse:.4f} | "
            f"MAE = {mae:.4f}"
        )

    return {
        "mean_pred": reg_means,
        "uncertainty": reg_vars,
        "y_true": reg_trues
    }


# =========================
# 10. 分类任务调用示例
# =========================
UQ_CFG["task"] = "cls"
UQ_CFG["eval_split"] = "all"   # 也可以改成 "val" 或 "test"

cls_results = evaluate_uncertainty_classification(ds_cls, UQ_CFG, device)

print("-----------------------------------------------------------------------------------------")

# =========================
# 11. 回归任务调用示例
# =========================
UQ_CFG["task"] = "reg"
UQ_CFG["eval_split"] = "all"  # 也可以改成 "val" 或 "test"

reg_results = evaluate_uncertainty_regression(ds_reg, UQ_CFG, device)

成功加载模型：saved_models\gin_cls_seed42_bs64_hd128_L4_do0.2_lr0.001_wd1e-05_ep100.pt
Checkpoint 中记录的超参数：
  task: cls
  seed: 42
  batch_size_train: 64
  batch_size_eval: 256
  hidden_dim: 128
  num_layers: 4
  dropout: 0.2
  lr: 0.001
  weight_decay: 1e-05
  epochs: 100
  model_dir: saved_models
  task_name: gin_cls

=== 分类任务：按不确定度分为 5 组 ===
第 01 组 | 不确定度范围 [0.000000, 0.000003] | 样本数 = 407 | 准确率 = 0.9902
第 02 组 | 不确定度范围 [0.000003, 0.000100] | 样本数 = 407 | 准确率 = 0.9853
第 03 组 | 不确定度范围 [0.000101, 0.001208] | 样本数 = 407 | 准确率 = 0.9803
第 04 组 | 不确定度范围 [0.001226, 0.005374] | 样本数 = 407 | 准确率 = 0.8919
第 05 组 | 不确定度范围 [0.005409, 0.095326] | 样本数 = 411 | 准确率 = 0.7737
-----------------------------------------------------------------------------------------
成功加载模型：saved_models\gin_reg_seed42_bs64_hd128_L4_do0.2_lr0.001_wd1e-05_ep100.pt
Checkpoint 中记录的超参数：
  task: reg
  seed: 42
  batch_size_train: 64
  batch_size_eval: 256
  hidden_dim: 128
  num_layers: 4
  dropout: 0.2
  lr: 0.001
  weight_decay: 1e-05

### （3） 按“预测置信度”最高和最低各看top-k个分子预测结果

In [24]:
# =========================
# 0. 不确定度分析参数区
# =========================
ANALYSIS_CFG = {
    "eval_split": "all",   # 可选: "train" / "val" / "test" / "all"
    "top_k": 20             # 展示前多少个极端样本
}


# =========================
# 1. 根据划分方式选择数据子集
# =========================
def get_selected_dataset(dataset, split="test", seed=42):
    """
    根据 split 选择要分析的数据集

    返回：
    - selected_dataset : 被选中的数据子集
    - original_indices : 该子集中每个样本在原始完整数据集中的编号
    """
    if split == "all":
        selected_dataset = dataset
        original_indices = list(range(len(dataset)))
        return selected_dataset, original_indices

    train_set, val_set, test_set = split_dataset(dataset, seed=seed)

    if split == "train":
        selected_dataset = train_set
    elif split == "val":
        selected_dataset = val_set
    elif split == "test":
        selected_dataset = test_set
    else:
        raise ValueError(f"未知的数据划分方式：{split}")

    if hasattr(selected_dataset, "indices"):
        original_indices = list(selected_dataset.indices)
    else:
        original_indices = list(range(len(selected_dataset)))

    return selected_dataset, original_indices


# =========================
# 2. 收集分类任务中每个分子的 MC-Dropout 结果
# =========================
def collect_mc_dropout_classification_results(
    dataset,
    model,
    device,
    split="test",
    seed=42,
    T=50,
    batch_size=64,
    pos_class=1,
    use_total_uncertainty=True
):
    """
    收集分类任务中每个分子的 MC-Dropout 结果
    """
    selected_dataset, original_indices = get_selected_dataset(
        dataset, split=split, seed=seed
    )

    loader = DataLoader(selected_dataset, batch_size=batch_size, shuffle=False)

    all_rows = []
    subset_offset = 0

    for batch in loader:
        batch = batch.to(device)

        mean_prob, var_prob = mc_dropout_predict_classification(model, batch, T=T)

        mean_prob_np = mean_prob.detach().cpu().numpy()
        pred_label = mean_prob.argmax(dim=-1).detach().cpu().numpy()
        true_label = batch.y.view(-1).detach().cpu().numpy()

        if use_total_uncertainty:
            uncertainty = var_prob.sum(dim=-1).detach().cpu().numpy()
        else:
            uncertainty = var_prob[:, pos_class].detach().cpu().numpy()

        batch_size_now = len(true_label)

        for i in range(batch_size_now):
            subset_id = subset_offset + i
            dataset_id = original_indices[subset_id]

            row = {
                "subset_id": subset_id,
                "dataset_id": int(dataset_id),
                "pred_prob_0": float(mean_prob_np[i, 0]),
                "pred_prob_1": float(mean_prob_np[i, 1]),
                "pred_prob_pos": float(mean_prob_np[i, pos_class]),
                "pred_label": int(pred_label[i]),
                "true_label": int(true_label[i]),
                "uncertainty": float(uncertainty[i]),
                "correct": int(pred_label[i] == true_label[i]),
                "eval_split": split
            }
            all_rows.append(row)

        subset_offset += batch_size_now

    df = pd.DataFrame(all_rows)

    print(f"\n当前分类分析数据集：{split}")
    print(f"样本总数：{len(df)}")

    return df


# =========================
# 3. 展示分类任务中不确定度最极端的样本
# =========================
def show_extreme_uncertainty_cases_classification(df, top_k=10):
    """
    展示分类任务中不确定度最低/最高的前 top_k 个分子
    """
    df_low_unc = df.sort_values("uncertainty", ascending=True).head(top_k).copy()
    df_high_unc = df.sort_values("uncertainty", ascending=False).head(top_k).copy()

    show_cols = [
        "dataset_id",
        "pred_prob_0",
        "pred_prob_1",
        "pred_prob_pos",
        "pred_label",
        "true_label",
        "uncertainty",
        "correct",
        "eval_split",
    ]

    print("\n==============================")
    print(f"【分类】不确定度最低的前 {top_k} 个分子")
    print("==============================")
    print(df_low_unc[show_cols].to_string(index=False))

    print("\n==============================")
    print(f"【分类】不确定度最高的前 {top_k} 个分子")
    print("==============================")
    print(df_high_unc[show_cols].to_string(index=False))

    print("\n--- 分类任务概要统计 ---")
    print(f"最低不确定度组准确率 = {df_low_unc['correct'].mean():.4f}")
    print(f"最高不确定度组准确率 = {df_high_unc['correct'].mean():.4f}")

    return df_low_unc, df_high_unc


# =========================
# 4. 收集回归任务中每个分子的 MC-Dropout 结果
# =========================
def collect_mc_dropout_regression_results(
    dataset,
    model,
    device,
    split="test",
    seed=42,
    T=50,
    batch_size=64
):
    """
    收集回归任务中每个分子的 MC-Dropout 结果

    返回：
    DataFrame，其中包含：
    - subset_id    : 当前所选数据子集中的编号
    - dataset_id   : 原始完整数据集中的编号
    - pred_value   : MC-Dropout 平均预测值
    - true_value   : 真实值
    - abs_error    : 绝对误差
    - uncertainty  : 不确定度（预测方差）
    - eval_split   : 当前使用的数据划分名称
    """
    selected_dataset, original_indices = get_selected_dataset(
        dataset, split=split, seed=seed
    )

    loader = DataLoader(selected_dataset, batch_size=batch_size, shuffle=False)

    all_rows = []
    subset_offset = 0

    for batch in loader:
        batch = batch.to(device)

        mean_pred, var_pred = mc_dropout_predict_regression(model, batch, T=T)

        mean_pred_np = mean_pred.detach().cpu().numpy()
        true_value_np = batch.y.view(-1).detach().cpu().numpy()
        uncertainty_np = var_pred.detach().cpu().numpy()

        batch_size_now = len(true_value_np)

        for i in range(batch_size_now):
            subset_id = subset_offset + i
            dataset_id = original_indices[subset_id]

            row = {
                "subset_id": subset_id,
                "dataset_id": int(dataset_id),
                "pred_value": float(mean_pred_np[i]),
                "true_value": float(true_value_np[i]),
                "abs_error": float(abs(mean_pred_np[i] - true_value_np[i])),
                "uncertainty": float(uncertainty_np[i]),
                "eval_split": split
            }
            all_rows.append(row)

        subset_offset += batch_size_now

    df = pd.DataFrame(all_rows)

    print(f"\n当前回归分析数据集：{split}")
    print(f"样本总数：{len(df)}")

    return df


# =========================
# 5. 展示回归任务中不确定度最极端的样本
#    这里看 MAE，不看准确率
# =========================
def show_extreme_uncertainty_cases_regression(df, top_k=10):
    """
    展示回归任务中不确定度最低/最高的前 top_k 个分子
    汇总指标使用 MAE
    """
    df_low_unc = df.sort_values("uncertainty", ascending=True).head(top_k).copy()
    df_high_unc = df.sort_values("uncertainty", ascending=False).head(top_k).copy()

    show_cols = [
        "dataset_id",
        "pred_value",
        "true_value",
        "abs_error",
        "uncertainty",
        "eval_split",
    ]

    print("\n==============================")
    print(f"【回归】不确定度最低的前 {top_k} 个分子")
    print("==============================")
    print(df_low_unc[show_cols].to_string(index=False))

    print("\n==============================")
    print(f"【回归】不确定度最高的前 {top_k} 个分子")
    print("==============================")
    print(df_high_unc[show_cols].to_string(index=False))

    print("\n--- 回归任务概要统计 ---")
    print(f"最低不确定度组 MAE = {df_low_unc['abs_error'].mean():.4f}")
    print(f"最高不确定度组 MAE = {df_high_unc['abs_error'].mean():.4f}")

    return df_low_unc, df_high_unc


# =========================
# 6. 调用示例：分类任务
# =========================
cfg_cls = copy.deepcopy(UQ_CFG)
cfg_cls["task"] = "cls"

model_cls, ckpt_cls = load_trained_model("cls", ds_cls, cfg_cls, device)

df_cls = collect_mc_dropout_classification_results(
    dataset=ds_cls,
    model=model_cls,
    device=device,
    split=ANALYSIS_CFG["eval_split"],
    seed=cfg_cls["seed"],
    T=cfg_cls["mc_T"],
    batch_size=cfg_cls["batch_size_eval"],
    pos_class=1,
    use_total_uncertainty=True
)

df_cls_low, df_cls_high = show_extreme_uncertainty_cases_classification(
    df_cls,
    top_k=ANALYSIS_CFG["top_k"]
)


# =========================
# 7. 调用示例：回归任务
# =========================
cfg_reg = copy.deepcopy(UQ_CFG)
cfg_reg["task"] = "reg"

model_reg, ckpt_reg = load_trained_model("reg", ds_reg, cfg_reg, device)

df_reg = collect_mc_dropout_regression_results(
    dataset=ds_reg,
    model=model_reg,
    device=device,
    split=ANALYSIS_CFG["eval_split"],
    seed=cfg_reg["seed"],
    T=cfg_reg["mc_T"],
    batch_size=cfg_reg["batch_size_eval"]
)

df_reg_low, df_reg_high = show_extreme_uncertainty_cases_regression(
    df_reg,
    top_k=ANALYSIS_CFG["top_k"]
)

成功加载模型：saved_models\gin_cls_seed42_bs64_hd128_L4_do0.2_lr0.001_wd1e-05_ep100.pt
Checkpoint 中记录的超参数：
  task: cls
  seed: 42
  batch_size_train: 64
  batch_size_eval: 256
  hidden_dim: 128
  num_layers: 4
  dropout: 0.2
  lr: 0.001
  weight_decay: 1e-05
  epochs: 100
  model_dir: saved_models
  task_name: gin_cls

当前分类分析数据集：all
样本总数：2039

【分类】不确定度最低的前 20 个分子
 dataset_id  pred_prob_0  pred_prob_1  pred_prob_pos  pred_label  true_label  uncertainty  correct eval_split
       1435 3.631640e-10     1.000000       1.000000           1           1 6.731588e-18        1        all
        431 9.431771e-08     1.000000       1.000000           1           1 1.236999e-13        1        all
       1520 1.393961e-07     1.000000       1.000000           1           1 2.582854e-13        1        all
       1343 1.328594e-07     1.000000       1.000000           1           1 5.680061e-13        1        all
       1933 1.384617e-06     0.999999       0.999999           1           1 7.493586e-12  

### (4) EDL不确定性应用于分类任务

In [25]:
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split
from torch_geometric.loader import DataLoader


# =========================
# 0. EDL 参数区
#   主要修改这里
# =========================
EDL_CFG = {
    "task": "cls",                        # 当前仅支持分类
    "seed": 42,                           # 随机种子
    "batch_size_train": 64,               # 训练 batch 大小
    "batch_size_eval": 256,               # 评估 batch 大小
    "hidden_dim": 128,                    # 隐藏层维度
    "num_layers": 4,                      # GINConv 层数
    "dropout": 0.2,                       # Dropout 概率
    "num_classes": 2,                     # 分类数
    "lr": 1e-3,                           # 学习率
    "weight_decay": 1e-5,                 # 权重衰减
    "epochs": 100,                         # 训练轮数
    "anneal_epochs": 10,                  # KL 退火轮数
    "kl_scale": 1e-2,                     # KL 项权重
    "model_dir": "saved_models",          # 模型保存目录
    "eval_split": "test"                  # 评估数据划分："train" / "val" / "test" / "all"
}


# =========================
# 1. 数据集划分函数
# =========================
def split_dataset(dataset, train_ratio=0.8, val_ratio=0.1, seed=42):
    """
    将数据集划分为训练集 / 验证集 / 测试集
    """
    n = len(dataset)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    n_test = n - n_train - n_val

    generator = torch.Generator().manual_seed(seed)
    train_set, val_set, test_set = random_split(
        dataset,
        [n_train, n_val, n_test],
        generator=generator
    )
    return train_set, val_set, test_set


# =========================
# 2. 根据超参数生成模型文件名
# =========================
def make_model_name(task_name, hp):
    """
    根据超参数生成模型文件名
    """
    return (
        f"{task_name}"
        f"_seed{hp['seed']}"
        f"_bs{hp['batch_size_train']}"
        f"_hd{hp['hidden_dim']}"
        f"_L{hp['num_layers']}"
        f"_do{hp['dropout']}"
        f"_lr{hp['lr']}"
        f"_wd{hp['weight_decay']}"
        f"_ep{hp['epochs']}"
        f"_kl{hp['kl_scale']}"
        f"_an{hp['anneal_epochs']}"
        f".pt"
    )


# =========================
# 3. EDL 的核心函数
# =========================
def dirichlet_kl(alpha, prior=None):
    """
    计算 Dirichlet(alpha) 与先验 Dirichlet(prior) 的 KL 散度
    默认先验为均匀 Dirichlet，即 prior = 1
    """
    if prior is None:
        prior = torch.ones_like(alpha)

    sum_alpha = alpha.sum(dim=-1, keepdim=True)
    sum_prior = prior.sum(dim=-1, keepdim=True)

    lnB_alpha = torch.lgamma(sum_alpha) - torch.lgamma(alpha).sum(dim=-1, keepdim=True)
    lnB_prior = torch.lgamma(sum_prior) - torch.lgamma(prior).sum(dim=-1, keepdim=True)

    digamma_sum = torch.digamma(sum_alpha)
    digamma_alpha = torch.digamma(alpha)

    kl = (alpha - prior) * (digamma_alpha - digamma_sum)
    kl = kl.sum(dim=-1, keepdim=True) + lnB_alpha - lnB_prior
    return kl.squeeze(-1)


def edl_ce_loss(alpha, y_onehot):
    """
    EDL 分类拟合损失：
    用 Dirichlet 分布的期望对数概率替代普通交叉熵
    """
    S = alpha.sum(dim=-1, keepdim=True)
    E_log_p = torch.digamma(alpha) - torch.digamma(S)
    return -(y_onehot * E_log_p).sum(dim=-1)


# =========================
# 4. EDL-Dirichlet 分类模型
# =========================
class GINDirichletClassifier(nn.Module):
    """
    GIN + Dirichlet 分类头：输出 Dirichlet 参数 alpha
    """
    def __init__(self, in_dim, hidden_dim=128, num_layers=4, dropout=0.2, num_classes=2):
        super().__init__()
        self.gnn = GINBackbone(in_dim, hidden_dim, num_layers, dropout)
        self.evidence_head = MLP(hidden_dim, hidden_dim, num_classes, dropout)

    def forward(self, data):
        g = self.gnn(data.x, data.edge_index, data.batch)    # 图级表示
        evidence = F.softplus(self.evidence_head(g))         # evidence >= 0
        alpha = evidence + 1.0                               # alpha > 1
        return alpha


# =========================
# 5. 根据配置构建模型
# =========================
def build_model(task, dataset, cfg, device):
    """
    根据任务类型构建模型
    """
    if task == "cls":
        model = GINDirichletClassifier(
            in_dim=dataset.num_features,
            hidden_dim=cfg["hidden_dim"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"],
            num_classes=cfg["num_classes"]
        ).to(device)
    else:
        raise ValueError(f"EDL 当前只实现分类任务，收到 task={task}")

    return model


# =========================
# 6. 数据加载器
# =========================
def get_data_loaders(dataset, cfg):
    """
    返回 train / val / test loader
    """
    train_set, val_set, test_set = split_dataset(dataset, seed=cfg["seed"])

    train_loader = DataLoader(
        train_set,
        batch_size=cfg["batch_size_train"],
        shuffle=True
    )
    val_loader = DataLoader(
        val_set,
        batch_size=cfg["batch_size_eval"],
        shuffle=False
    )
    test_loader = DataLoader(
        test_set,
        batch_size=cfg["batch_size_eval"],
        shuffle=False
    )
    return train_loader, val_loader, test_loader


def get_eval_loader(dataset, cfg):
    """
    根据 eval_split 选择评估集
    """
    train_set, val_set, test_set = split_dataset(dataset, seed=cfg["seed"])

    split = cfg["eval_split"]
    if split == "train":
        eval_set = train_set
    elif split == "val":
        eval_set = val_set
    elif split == "test":
        eval_set = test_set
    elif split == "all":
        eval_set = dataset
    else:
        raise ValueError(f"未知的数据划分方式：{split}")

    loader = DataLoader(
        eval_set,
        batch_size=cfg["batch_size_eval"],
        shuffle=False
    )
    return loader


# =========================
# 7. 单轮训练函数
# =========================
def train_one_epoch_edl(model, loader, optimizer, device, epoch, cfg):
    """
    训练一个 epoch：
      总损失 = 拟合损失 + KL 正则
    """
    model.train()
    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)
        y = batch.y.view(-1).long()
        y_onehot = F.one_hot(y, num_classes=cfg["num_classes"]).float()

        alpha = model(batch)
        loss_fit = edl_ce_loss(alpha, y_onehot).mean()
        loss_kl = dirichlet_kl(alpha).mean()

        w = min(1.0, epoch / float(cfg["anneal_epochs"]))
        loss = loss_fit + w * cfg["kl_scale"] * loss_kl

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.detach().item() * batch.num_graphs

    return total_loss / max(len(loader.dataset), 1)


# =========================
# 8. 验证函数
# =========================
@torch.no_grad()
def evaluate_edl_accuracy(model, loader, device):
    """
    在验证集或测试集上计算准确率
    """
    model.eval()
    correct = 0
    total = 0

    for batch in loader:
        batch = batch.to(device)
        p_mean, u = edl_predict_classification(model, batch)

        pred = p_mean.argmax(dim=-1)
        y = batch.y.view(-1).long()

        correct += int((pred == y).sum())
        total += y.numel()

    return correct / max(total, 1)


# =========================
# 9. 训练并保存模型
# =========================
def train_and_save_model(task, dataset, cfg, device):
    """
    根据配置训练模型，并保存最佳 checkpoint
    """
    if task != "cls":
        raise ValueError(f"EDL 当前只实现分类任务，收到 task={task}")

    os.makedirs(cfg["model_dir"], exist_ok=True)

    train_loader, val_loader, test_loader = get_data_loaders(dataset, cfg)
    model = build_model(task, dataset, cfg, device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"]
    )

    task_name = "gin_edl_dirichlet_cls"
    save_name = make_model_name(task_name, cfg)
    save_path = os.path.join(cfg["model_dir"], save_name)

    best_val_acc = -1.0
    best_epoch = -1

    for epoch in range(1, cfg["epochs"] + 1):
        train_loss = train_one_epoch_edl(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            epoch=epoch,
            cfg=cfg
        )

        val_acc = evaluate_edl_accuracy(model, val_loader, device)
        if epoch%10==0:
            print(
                f"第 {epoch:03d} 轮 | "
                f"训练损失 = {train_loss:.4f} | "
                f"验证集准确率 = {val_acc:.4f}"
            )
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch

            ckpt = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_acc": best_val_acc,
                "hyperparameters": dict(cfg)
            }
            torch.save(ckpt, save_path)

    print(f"\n分类模型训练完成")
    print(f"最佳验证集准确率 = {best_val_acc:.4f}")
    print(f"模型已保存到：{save_path}")

    return model


# =========================
# 10. 加载已经训练好的模型
# =========================
def load_trained_model(task, dataset, cfg, device):
    """
    根据超参数自动定位并加载已训练模型
    """
    if task != "cls":
        raise ValueError(f"EDL 当前只实现分类任务，收到 task={task}")

    task_name = "gin_edl_dirichlet_cls"
    save_name = make_model_name(task_name, cfg)
    save_path = os.path.join(cfg["model_dir"], save_name)

    if not os.path.exists(save_path):
        raise FileNotFoundError(f"找不到模型文件：{save_path}")

    model = build_model(task, dataset, cfg, device)

    ckpt = torch.load(save_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])

    print(f"成功加载模型：{save_path}")
    if "best_val_acc" in ckpt:
        print(f"Checkpoint 最佳验证集准确率：{ckpt['best_val_acc']:.4f}")
    if "hyperparameters" in ckpt:
        print("Checkpoint 中记录的超参数：")
        for k, v in ckpt["hyperparameters"].items():
            print(f"  {k}: {v}")

    return model, ckpt


# =========================
# 11. EDL 分类预测函数
# =========================
@torch.no_grad()
def edl_predict_classification(model, data):
    """
    分类任务下的 EDL 预测

    输入：
      - model: EDL 分类模型，输出形状为 [B, K] 的 alpha
      - data : 一个 batch 的图数据

    输出：
      - p_mean: 平均类别概率，[B, K]
      - u     : 总不确定度，[B]
    """
    model.eval()
    alpha = model(data)                              # [B, K]
    S = alpha.sum(dim=-1, keepdim=True)             # [B, 1]
    p_mean = alpha / S                              # [B, K]
    u = (alpha.size(-1) / S).squeeze(-1)            # [B]
    return p_mean, u


# =========================
# 12. 加载模型并评估不确定度
# =========================
def evaluate_uncertainty_classification(dataset, cfg, device):
    """
    对 EDL 分类任务进行不确定度评估
    """
    model, ckpt = load_trained_model("cls", dataset, cfg, device)
    loader = get_eval_loader(dataset, cfg)

    cls_means = []
    cls_uncs = []
    cls_trues = []

    for batch in loader:
        batch = batch.to(device)

        p_mean, u = edl_predict_classification(model, batch)

        y_true = batch.y.view(-1).detach().cpu().numpy()

        cls_means.append(p_mean.detach().cpu().numpy())
        cls_uncs.append(u.detach().cpu().numpy())
        cls_trues.append(y_true)

    cls_means = np.concatenate(cls_means, axis=0)   # [N, K]
    cls_uncs = np.concatenate(cls_uncs, axis=0)     # [N]
    cls_trues = np.concatenate(cls_trues, axis=0)   # [N]

    cls_sorted_idx = np.argsort(cls_uncs)
    n_cls = len(cls_sorted_idx)
    cls_group_size = n_cls // 5

    print("\n=== EDL 分类任务：按不确定度分为 5 组 ===")
    for i in range(5):
        start = i * cls_group_size
        end = n_cls if i == 4 else (i + 1) * cls_group_size
        idxs = cls_sorted_idx[start:end]

        preds = cls_means[idxs].argmax(axis=-1)
        trues = cls_trues[idxs]
        acc = (preds == trues).mean()

        print(
            f"第 {i+1:02d} 组 | "
            f"不确定度范围 [{cls_uncs[idxs].min():.6f}, {cls_uncs[idxs].max():.6f}] | "
            f"平均不确定度 = {cls_uncs[idxs].mean():.6f} | "
            f"样本数 = {len(idxs)} | "
            f"准确率 = {acc:.4f}"
        )

    return {
        "mean_prob": cls_means,
        "uncertainty": cls_uncs,
        "y_true": cls_trues
    }


# =========================
# 13. 训练代码
#   先训练并保存模型
# =========================
EDL_CFG["task"] = "cls"
trained_model = train_and_save_model("cls", ds_cls, EDL_CFG, device)


# =========================
# 14. 不确定度评估调用示例
# =========================
EDL_CFG["eval_split"] = "all"    # 也可以改成 "val" 或 "test"

cls_results = evaluate_uncertainty_classification(ds_cls, EDL_CFG, device)

第 010 轮 | 训练损失 = 0.3697 | 验证集准确率 = 0.8424
第 020 轮 | 训练损失 = 0.3269 | 验证集准确率 = 0.8276
第 030 轮 | 训练损失 = 0.3076 | 验证集准确率 = 0.8325
第 040 轮 | 训练损失 = 0.2809 | 验证集准确率 = 0.8571
第 050 轮 | 训练损失 = 0.2706 | 验证集准确率 = 0.8522
第 060 轮 | 训练损失 = 0.2603 | 验证集准确率 = 0.8276
第 070 轮 | 训练损失 = 0.2565 | 验证集准确率 = 0.8571
第 080 轮 | 训练损失 = 0.2358 | 验证集准确率 = 0.8374
第 090 轮 | 训练损失 = 0.2249 | 验证集准确率 = 0.8424
第 100 轮 | 训练损失 = 0.2189 | 验证集准确率 = 0.8621

分类模型训练完成
最佳验证集准确率 = 0.8768
模型已保存到：saved_models\gin_edl_dirichlet_cls_seed42_bs64_hd128_L4_do0.2_lr0.001_wd1e-05_ep100_kl0.01_an10.pt
成功加载模型：saved_models\gin_edl_dirichlet_cls_seed42_bs64_hd128_L4_do0.2_lr0.001_wd1e-05_ep100_kl0.01_an10.pt
Checkpoint 最佳验证集准确率：0.8768
Checkpoint 中记录的超参数：
  task: cls
  seed: 42
  batch_size_train: 64
  batch_size_eval: 256
  hidden_dim: 128
  num_layers: 4
  dropout: 0.2
  num_classes: 2
  lr: 0.001
  weight_decay: 1e-05
  epochs: 100
  anneal_epochs: 10
  kl_scale: 0.01
  model_dir: saved_models
  eval_split: test

=== EDL 分类任务：按不确定度分为 5 组 ==

### (8) EDL不确定性应用于回归任务

In [26]:
# =========================
# 0. EDL-NIG 不确定度评估参数区
#   主要修改这里
# =========================
EDL_REG_CFG = {
    "task": "reg",                        # 当前任务类型：回归
    "seed": 42,                           # 随机种子
    "batch_size_train": 64,               # 训练 batch 大小
    "batch_size_eval": 256,               # 评估 batch 大小
    "hidden_dim": 128,                    # 隐藏层维度
    "num_layers": 4,                      # GINConv 层数
    "dropout": 0.2,                       # Dropout 概率
    "lr": 1e-3,                           # 学习率
    "weight_decay": 1e-5,                 # 权重衰减
    "epochs": 100,                         # 训练轮数
    "lam": 1e-2,                          # NIG 证据正则项系数
    "model_dir": "saved_models",          # 模型保存目录
    "eval_split": "test"                  # 评估数据划分："train" / "val" / "test" / "all"
}


# =========================
# 1. 数据集划分函数
# =========================
def split_dataset(dataset, train_ratio=0.8, val_ratio=0.1, seed=42):
    """
    将数据集划分为训练集 / 验证集 / 测试集
    """
    n = len(dataset)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    n_test = n - n_train - n_val

    generator = torch.Generator().manual_seed(seed)
    train_set, val_set, test_set = random_split(
        dataset,
        [n_train, n_val, n_test],
        generator=generator
    )
    return train_set, val_set, test_set


# =========================
# 2. 根据超参数生成模型文件名
# =========================
def make_model_name(task_name, hp):
    """
    根据超参数生成模型文件名
    """
    return (
        f"{task_name}"
        f"_seed{hp['seed']}"
        f"_bs{hp['batch_size_train']}"
        f"_hd{hp['hidden_dim']}"
        f"_L{hp['num_layers']}"
        f"_do{hp['dropout']}"
        f"_lr{hp['lr']}"
        f"_wd{hp['weight_decay']}"
        f"_ep{hp['epochs']}"
        f"_lam{hp['lam']}"
        f".pt"
    )


# =========================
# 3. NIG 回归模型
# =========================
class GINNIGRegressor(nn.Module):
    """
    GIN + NIG 回归头：输出 Normal-Inverse-Gamma 分布参数
    """
    def __init__(self, in_dim, hidden_dim=128, num_layers=4, dropout=0.2):
        super().__init__()
        self.gnn = GINBackbone(in_dim, hidden_dim, num_layers, dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 4)   # 输出 mu, v, alpha, beta
        )

    def forward(self, data):
        g = self.gnn(data.x, data.edge_index, data.batch)   # 图级表示
        out = self.fc(g)                                    # [B, 4]

        mu = out[:, 0]                                      # 预测均值
        v = F.softplus(out[:, 1]) + 1e-6                    # v > 0
        alpha = F.softplus(out[:, 2]) + 1.0                 # alpha > 1
        beta = F.softplus(out[:, 3]) + 1e-6                 # beta > 0

        return mu, v, alpha, beta


# =========================
# 4. NIG 损失函数
# =========================
def nig_nll(y, mu, v, alpha, beta):
    """
    NIG 分布的负对数似然项
    """
    two_beta_v = 2.0 * beta * (1.0 + v)
    nll = (
        0.5 * torch.log(torch.pi / v)
        - alpha * torch.log(two_beta_v)
        + (alpha + 0.5) * torch.log(v * (y - mu) ** 2 + two_beta_v)
        + torch.lgamma(alpha)
        - torch.lgamma(alpha + 0.5)
    )
    return nll


def nig_reg(y, mu, v, alpha, beta):
    """
    误差越大时，对过强证据施加更大惩罚
    """
    return torch.abs(y - mu) * (2.0 * v + alpha)


# =========================
# 5. 根据任务构建模型
# =========================
def build_model(task, dataset, cfg, device):
    """
    根据任务类型构建模型
    """
    if task == "reg":
        model = GINNIGRegressor(
            in_dim=dataset.num_features,
            hidden_dim=cfg["hidden_dim"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"]
        ).to(device)
    else:
        raise ValueError(f"EDL-NIG 当前只实现回归任务，收到 task={task}")

    return model


# =========================
# 6. 数据加载器
# =========================
def get_data_loaders(dataset, cfg):
    """
    返回 train / val / test loader
    """
    train_set, val_set, test_set = split_dataset(dataset, seed=cfg["seed"])

    train_loader = DataLoader(
        train_set,
        batch_size=cfg["batch_size_train"],
        shuffle=True
    )
    val_loader = DataLoader(
        val_set,
        batch_size=cfg["batch_size_eval"],
        shuffle=False
    )
    test_loader = DataLoader(
        test_set,
        batch_size=cfg["batch_size_eval"],
        shuffle=False
    )
    return train_loader, val_loader, test_loader


def get_eval_loader(dataset, cfg):
    """
    根据 eval_split 选择评估数据
    """
    train_set, val_set, test_set = split_dataset(dataset, seed=cfg["seed"])

    split = cfg["eval_split"]
    if split == "train":
        eval_set = train_set
    elif split == "val":
        eval_set = val_set
    elif split == "test":
        eval_set = test_set
    elif split == "all":
        eval_set = dataset
    else:
        raise ValueError(f"未知的数据划分方式：{split}")

    loader = DataLoader(
        eval_set,
        batch_size=cfg["batch_size_eval"],
        shuffle=False
    )
    return loader


# =========================
# 7. 单轮训练函数
# =========================
def train_one_epoch_edl_nig(model, loader, optimizer, device, cfg):
    """
    训练一个 epoch：
      总损失 = NIG 负对数似然 + lam * 正则项
    """
    model.train()
    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)
        y = batch.y.view(-1).float()

        mu, v, alpha, beta = model(batch)

        loss = (
            nig_nll(y, mu, v, alpha, beta).mean()
            + cfg["lam"] * nig_reg(y, mu, v, alpha, beta).mean()
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.detach().item() * batch.num_graphs

    return total_loss / max(len(loader.dataset), 1)


# =========================
# 8. 回归预测函数
# =========================
@torch.no_grad()
def edl_predict_regression(model, data):
    """
    回归任务下的 EDL-NIG 预测

    输入：
      - model: 输出 mu, v, alpha, beta 的 NIG 回归模型
      - data : 一个 batch 图数据

    输出：
      - mu  : [B]，预测均值
      - epistemic : [B]，认知不确定性
    """
    model.eval()

    mu, v, alpha, beta = model(data)

    # 由 NIG 参数导出预测方差
    epistemic = beta / (v * (alpha - 1.0) + 1e-8)

    return mu, epistemic


# =========================
# 9. 验证函数
# =========================
@torch.no_grad()
def evaluate_regression_metrics(model, loader, device):
    """
    在验证集或测试集上计算 RMSE / MAE
    """
    model.eval()

    ys = []
    ps = []

    for batch in loader:
        batch = batch.to(device)
        mu, var = edl_predict_regression(model, batch)

        ys.append(batch.y.view(-1).float().detach().cpu())
        ps.append(mu.detach().cpu())

    y = torch.cat(ys)
    p = torch.cat(ps)

    rmse = torch.sqrt(torch.mean((p - y) ** 2)).item()
    mae = torch.mean(torch.abs(p - y)).item()

    return rmse, mae


# =========================
# 10. 训练并保存模型
# =========================
def train_and_save_model(task, dataset, cfg, device):
    """
    根据配置训练模型，并保存最佳 checkpoint
    """
    if task != "reg":
        raise ValueError(f"EDL-NIG 当前只实现回归任务，收到 task={task}")

    os.makedirs(cfg["model_dir"], exist_ok=True)

    train_loader, val_loader, test_loader = get_data_loaders(dataset, cfg)
    model = build_model(task, dataset, cfg, device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"]
    )

    task_name = "gin_edl_nig_reg"
    save_name = make_model_name(task_name, cfg)
    save_path = os.path.join(cfg["model_dir"], save_name)

    best_val_rmse = float("inf")
    best_epoch = -1

    for epoch in range(1, cfg["epochs"] + 1):
        train_loss = train_one_epoch_edl_nig(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            cfg=cfg
        )

        val_rmse, val_mae = evaluate_regression_metrics(model, val_loader, device)

        if epoch%10==0:
            print(
                f"第 {epoch:03d} 轮 | "
                f"训练损失 = {train_loss:.4f} | "
                f"验证集 MAE = {val_mae:.4f} | "
                f"验证集 RMSE = {val_rmse:.4f}"
            )

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch

            ckpt = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_rmse": best_val_rmse,
                "hyperparameters": dict(cfg)
            }
            torch.save(ckpt, save_path)

    print(f"\n回归模型训练完成")
    print(f"最佳验证集 RMSE = {best_val_rmse:.4f}")
    print(f"模型已保存到：{save_path}")


    return model


# =========================
# 11. 加载已经训练好的模型
# =========================
def load_trained_model(task, dataset, cfg, device):
    """
    根据超参数自动定位并加载已训练模型
    """
    if task != "reg":
        raise ValueError(f"EDL-NIG 当前只实现回归任务，收到 task={task}")

    task_name = "gin_edl_nig_reg"
    save_name = make_model_name(task_name, cfg)
    save_path = os.path.join(cfg["model_dir"], save_name)

    if not os.path.exists(save_path):
        raise FileNotFoundError(f"找不到模型文件：{save_path}")

    model = build_model(task, dataset, cfg, device)

    ckpt = torch.load(save_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])

    print(f"成功加载模型：{save_path}")
    if "best_val_rmse" in ckpt:
        print(f"Checkpoint 最佳验证集 RMSE：{ckpt['best_val_rmse']:.4f}")
    if "hyperparameters" in ckpt:
        print("Checkpoint 中记录的超参数：")
        for k, v in ckpt["hyperparameters"].items():
            print(f"  {k}: {v}")

    return model, ckpt


# =========================
# 12. 回归任务：加载模型并评估不确定度
# =========================
def evaluate_uncertainty_regression(dataset, cfg, device):
    """
    对 EDL-NIG 回归任务进行不确定度评估
    """
    model, ckpt = load_trained_model("reg", dataset, cfg, device)
    loader = get_eval_loader(dataset, cfg)

    reg_means = []
    reg_epistemic = []
    reg_trues = []

    for batch in loader:
        batch = batch.to(device)

        mu, epistemic = edl_predict_regression(model, batch)

        reg_means.append(mu.detach().cpu().numpy())
        reg_epistemic.append(epistemic.detach().cpu().numpy())
        reg_trues.append(batch.y.view(-1).detach().cpu().numpy())

    reg_means = np.concatenate(reg_means, axis=0)
    reg_epistemic = np.concatenate(reg_epistemic, axis=0)
    reg_trues = np.concatenate(reg_trues, axis=0)

    reg_sorted_idx = np.argsort(reg_epistemic)
    n_reg = len(reg_sorted_idx)
    reg_group_size = n_reg // 5

    print("\n=== EDL-NIG 回归任务：按不确定度分为 5 组 ===")
    for i in range(5):
        start = i * reg_group_size
        end = n_reg if i == 4 else (i + 1) * reg_group_size
        idxs = reg_sorted_idx[start:end]

        preds = reg_means[idxs]
        trues = reg_trues[idxs]
        epistemic_ = reg_epistemic[idxs]

        rmse = np.sqrt(((preds - trues) ** 2).mean())
        mae = np.abs(preds - trues).mean()

        print(
            f"第 {i+1:02d} 组 | "
            f"不确定度范围 [{epistemic_.min():.6f}, {epistemic_.max():.6f}] | "
            f"平均不确定度 = {epistemic_.mean():.6f} | "
            f"样本数 = {len(idxs)} | "
            f"RMSE = {rmse:.4f} | "
            f"MAE = {mae:.4f}"
        )

    return {
        "mean_pred": reg_means,
        "uncertainty": reg_epistemic,
        "y_true": reg_trues
    }


# =========================
# 13. 训练代码
#   先训练并保存模型
# =========================
EDL_REG_CFG["task"] = "reg"
trained_model = train_and_save_model("reg", ds_reg, EDL_REG_CFG, device)


# =========================
# 14. 不确定度评估调用示例
# =========================
EDL_REG_CFG["eval_split"] = "all"   # 也可以改成 "val" 或 "test"
reg_results = evaluate_uncertainty_regression(ds_reg, EDL_REG_CFG, device)

第 010 轮 | 训练损失 = 1.5653 | 验证集 MAE = 1.0435 | 验证集 RMSE = 1.3059
第 020 轮 | 训练损失 = 1.4206 | 验证集 MAE = 1.0944 | 验证集 RMSE = 1.4322
第 030 轮 | 训练损失 = 1.3722 | 验证集 MAE = 1.0541 | 验证集 RMSE = 1.3925
第 040 轮 | 训练损失 = 1.3663 | 验证集 MAE = 1.0114 | 验证集 RMSE = 1.3175
第 050 轮 | 训练损失 = 1.2571 | 验证集 MAE = 1.1333 | 验证集 RMSE = 1.3920
第 060 轮 | 训练损失 = 1.2178 | 验证集 MAE = 0.6886 | 验证集 RMSE = 0.9325
第 070 轮 | 训练损失 = 1.2232 | 验证集 MAE = 0.8897 | 验证集 RMSE = 1.1805
第 080 轮 | 训练损失 = 1.2241 | 验证集 MAE = 0.7299 | 验证集 RMSE = 0.9462
第 090 轮 | 训练损失 = 1.2082 | 验证集 MAE = 0.8393 | 验证集 RMSE = 1.1238
第 100 轮 | 训练损失 = 1.1291 | 验证集 MAE = 0.7134 | 验证集 RMSE = 0.9543

回归模型训练完成
最佳验证集 RMSE = 0.8381
模型已保存到：saved_models\gin_edl_nig_reg_seed42_bs64_hd128_L4_do0.2_lr0.001_wd1e-05_ep100_lam0.01.pt
成功加载模型：saved_models\gin_edl_nig_reg_seed42_bs64_hd128_L4_do0.2_lr0.001_wd1e-05_ep100_lam0.01.pt
Checkpoint 最佳验证集 RMSE：0.8381
Checkpoint 中记录的超参数：
  task: reg
  seed: 42
  batch_size_train: 64
  batch_size_eval: 256
  hidden_dim: 128
  num_layers

### (9) 探究：
1. 高不确定性样本是否更容易预测错？<br>
请将样本按不确定性大小分组，比较各组的预测表现。思考：如果高不确定性组确实更容易出错，这说明了什么？<br>
2. 低不确定性是否一定意味着预测正确？<br>
请寻找“模型很确定但仍然预测错误”的样本。思考：为什么会出现“自信但错”的情况？这种情况在实际应用中危险吗？<br>
3. MC 采样次数T越大越好吗？<br>
请尝试不同的T，例如 10、30、50、100。观察：<br>
不确定性估计是否更稳定 <br>
运行时间是否明显增加 <br>
思考：为什么采样次数增加会提升稳定性，但也会增加计算成本？